# M09 lab solution: approval loop

**Brief.** Add a checkpointer and an interrupt before the answer node of your M08 graph, resume it after approval, and stream the steps.

The M08 graph is rebuilt here (classify, then `billing_answer` or `support_answer`), compiled with `MemorySaver` and `interrupt_before` both answer nodes. One run stops before answering, a reviewer approves it, and it resumes from the checkpoint. A second thread is rejected and never resumes. Every step is streamed with `stream_mode="updates"`.

Key: `EXPERIENTIALLABS_API_KEY`.

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

sys.path.insert(0, str(Path("../../../../src").resolve()))
load_dotenv()

DATA = Path("../data")

import uuid

from typing import Literal, TypedDict

import pandas as pd
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from pydantic import BaseModel, Field

from helpers.utils import get_experientiallabs_llm

In [ ]:
from lab_kit import load_handbook, load_jsonl, tfidf_retriever
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [c for p in pages for c in splitter.split_text(p["text"])]
print(len(pages), "handbook files,", len(chunks), "chunks,", len(golden), "golden questions")

## 1. The M08 graph, rebuilt inline

In [ ]:
class Route(BaseModel):
    category: Literal["billing", "technical", "general"]

class Answer(BaseModel):
    answer: str = Field(description="The answer, using only the passages")
    sources: list[str] = Field(description="Handbook files the answer uses, such as 02_refunds.md")

class State(TypedDict, total=False):
    question: str
    category: str
    node: str
    answer: dict

BILLING_FILES = {"01_billing.md", "02_refunds.md", "12_sla_credits.md"}

def passages_from(sources: set[str] | None) -> list[tuple[str, str]]:
    return [(p["source"], c) for p in pages if sources is None or p["source"] in sources
            for c in splitter.split_text(p["text"])]

billing_passages = passages_from(BILLING_FILES)
all_passages = passages_from(None)
billing_search = tfidf_retriever([c for _, c in billing_passages], k=3)
support_search = tfidf_retriever([c for _, c in all_passages], k=3)
source_of = {c: s for s, c in all_passages}

llm = get_experientiallabs_llm()
classifier = llm.with_structured_output(Route)
answerer = llm.with_structured_output(Answer)

def classify(state: State) -> dict:
    route = classifier.invoke(
        "Classify this customer question about a cloud hosting company as billing (payments, invoices, "
        "refunds, credits), technical (servers, API, backups, regions, security) or general (anything else).\n\n"
        f"Question: {state['question']}")
    return {"category": route.category}

def route(state: State) -> str:
    return "billing_answer" if state["category"] == "billing" else "support_answer"

def answer_from(state: State, found: list[str], node: str) -> dict:
    context = "\n\n".join(f"[{source_of[c]}] {c}" for c in found)
    result = answerer.invoke(
        "Answer the customer's question using only these passages from the Larkspur handbook. "
        "List the file names you used as sources. If the passages do not answer it, say so.\n\n"
        f"{context}\n\nQuestion: {state['question']}")
    return {"node": node, "answer": result.model_dump()}

def billing_answer(state: State) -> dict:
    return answer_from(state, billing_search(state["question"]), "billing_answer")

def support_answer(state: State) -> dict:
    return answer_from(state, support_search(state["question"]), "support_answer")

builder = StateGraph(State)
builder.add_node("classify", classify)
builder.add_node("billing_answer", billing_answer)
builder.add_node("support_answer", support_answer)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", route, ["billing_answer", "support_answer"])
builder.add_edge("billing_answer", END)
builder.add_edge("support_answer", END)

## 2. Compile with a checkpointer and an interrupt before the answer

The checkpointer saves the state after every step under the `thread_id`; without it there is nothing to resume from.

In [ ]:
graph = builder.compile(checkpointer=MemorySaver(), interrupt_before=["billing_answer", "support_answer"])

def stream(inputs, config) -> None:
    for update in graph.stream(inputs, config, stream_mode="updates"):
        print(update)

## 3. A run that stops, is approved, and resumes

In [ ]:
question = next(g for g in golden if g["id"] == "g01")["question"]
approved_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

stream({"question": question}, approved_config)
paused = graph.get_state(approved_config)
print("paused before:", paused.next)
print("state so far:", paused.values)

In [ ]:
# The reviewer reads the paused state and approves. Approval here is a value the caller sets;
# in an app it comes from a person, and anything but an explicit yes means no.
decision = "approve"

if decision == "approve":
    stream(None, approved_config)  # None resumes from the checkpoint instead of starting over

final = graph.get_state(approved_config)
print("next:", final.next)
print("answer:", final.values.get("answer"))

## 4. A run that is rejected

A different `thread_id` has its own history. Nothing resumes it, so the answer node never runs.

In [ ]:
rejected_question = next(g for g in golden if g["id"] == "g05")["question"]
rejected_config = {"configurable": {"thread_id": str(uuid.uuid4())}}
stream({"question": rejected_question}, rejected_config)

decision = "reject"
rejected = graph.get_state(rejected_config)
print("next:", rejected.next, "| answer:", rejected.values.get("answer"))

## Results

In [ ]:
pd.DataFrame([
    {"thread": "approved", "question": question, "category": final.values.get("category"),
     "stopped before": paused.next, "after resume": final.next,
     "answered": final.values.get("answer") is not None,
     "checkpoints": len(list(graph.get_state_history(approved_config)))},
    {"thread": "rejected", "question": rejected_question, "category": rejected.values.get("category"),
     "stopped before": rejected.next, "after resume": rejected.next,
     "answered": rejected.values.get("answer") is not None,
     "checkpoints": len(list(graph.get_state_history(rejected_config)))},
])

## What this shows

- `interrupt_before` pauses between steps, after `classify` has written its category, so the reviewer sees the route before any answer is written.
- Resuming with `None` continues the saved thread; passing the question again would start a new run on the same thread.
- `stream_mode="updates"` prints one line per node with only the keys it changed, which is the progress view an approval screen needs.